# Interaction Finder Walkthrough

This notebook demonstrates `InteractionFinder`, a bootstrap-based pairwise interaction
detector for residual signal. It continues from the [V2 walkthrough](residual_signal_finder_v2_walkthrough.ipynb)
using the same UCI **Default of Credit Card Clients** dataset and the same base model.

## Workflow

1. Fit a base logistic regression and compute out-of-fold residuals.
2. Run `ResidualSignalFinderV2` to rank candidate features by univariate residual lift.
3. Select top features (those the V2 summary labels as credible signal).
4. Fit `InteractionFinder` on those features to find which *pairs* show interaction
   effects beyond their additive main effects.
5. Inspect `interaction_summary_` and `interaction_bootstrap_results_`.
6. Visualise the top-ranked interactions with `plot_interactions()`.

## What InteractionFinder detects

For each feature pair (A, B), `InteractionFinder` fits two ensemble models on residuals
across bootstrap splits:

- **Depth-1 (additive baseline)**: each tree is limited to depth 1, so the ensemble can
  learn the separate marginal effects of A and B but cannot split on both in a single
  tree — no A×B interaction term.
- **Depth-2 (interaction-capable)**: trees can split on A then B (or vice versa),
  capturing the interaction.

**Interaction lift** = R²(depth-2) − R²(depth-1). A positive lift means the pair
predicts residuals better together than their additive effects alone.

A **permuted-second-feature null** guards against false positives: feature B is shuffled
within the training and validation sets independently, breaking the real A×B relationship
while preserving B's marginal distribution. The `interaction_null_beat_rate` is the
fraction of bootstrap splits where the real lift exceeded this null lift.

## Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

for path in [PROJECT_ROOT / 'src', PROJECT_ROOT]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

PROJECT_ROOT

In [ ]:
import warnings

import matplotlib
%matplotlib inline

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

matplotlib.use('Agg')
import matplotlib.pyplot as plt

from pe_tools.signal_finder import InteractionFinder, ResidualSignalFinderV2

warnings.filterwarnings('ignore', category=ConvergenceWarning)
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

## Load data and build analysis sample

Same setup as the V2 walkthrough: a 2 500-row stratified sample from the full 30 000-row
UCI credit dataset. An `age_band` categorical feature is added to exercise categorical
handling in `InteractionFinder`.

In [ ]:
DATA_PATH = PROJECT_ROOT / 'data' / 'default_of_credit_card_clients.csv'
credit = pd.read_csv(DATA_PATH)
target_col = 'default_next_month'
base_feature_cols = [c for c in credit.columns if c != target_col]

analysis_frame = train_test_split(
    credit,
    train_size=2_500,
    stratify=credit[target_col],
    random_state=42,
)[0].sort_index()

X_base = analysis_frame[base_feature_cols].copy()
y_true = analysis_frame[target_col].astype(float).rename(target_col)

default_rate = float(y_true.mean())
class_weight = np.where(y_true.eq(1), 0.5 / default_rate, 0.5 / (1.0 - default_rate))
limit_weight = (X_base['limit_bal'] / X_base['limit_bal'].median()).clip(0.25, 4.0)
sample_weight = pd.Series(
    class_weight * limit_weight.to_numpy(), index=X_base.index, name='sample_weight'
)

X_candidates = X_base.copy()
X_candidates['age_band'] = pd.cut(
    X_base['age'], bins=[20, 30, 40, 50, 60, 80], include_lowest=True
).astype(str)

print(f'Rows: {len(analysis_frame):,}   Positive rate: {default_rate:.3f}')
print(f'Candidate features: {X_candidates.shape[1]}')

## Build out-of-fold base predictions

`InteractionFinder` expects pre-computed residuals. We first build OOF predictions from
a simple logistic regression so the residuals represent genuine out-of-sample errors.

In [ ]:
oof_pred = pd.Series(np.nan, index=X_base.index, name='oof_default_probability')
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(cv.split(X_base, y_true), start=1):
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1_000, solver='lbfgs'),
    )
    model.fit(
        X_base.iloc[train_idx],
        y_true.iloc[train_idx],
        logisticregression__sample_weight=sample_weight.iloc[train_idx],
    )
    oof_pred.iloc[val_idx] = model.predict_proba(X_base.iloc[val_idx])[:, 1]

residuals = (y_true - oof_pred).rename('residual')
print(f'OOF AUC: {roc_auc_score(y_true, oof_pred):.3f}')
print(f'Residual mean: {residuals.mean():.4f}   std: {residuals.std():.4f}')

## Step 1 — Screen features with ResidualSignalFinderV2

`InteractionFinder` is capped at 15 candidate features (`max_candidate_features=15`)
because it evaluates every pair: 15 features → C(15, 2) = 105 pairs × 50 bootstraps
= 5 250 model fits. Pre-screening with V2 reduces this to a credible candidate set.

We use `n_bootstraps=20` here for notebook speed. The V2 walkthrough uses the same
settings; see that notebook for a full explanation of each parameter.

In [ ]:
v2_finder = ResidualSignalFinderV2(
    screening_enabled=True,
    screening_model_type='random_forest',
    screening_top_k=15,
    screening_cv_folds=3,
    screening_n_repeats=1,
    univariate_model_type='random_forest',
    split_strategy='bootstrap',
    n_bootstraps=20,
    test_size=0.25,
    n_bins=20,
    random_state=42,
    use_sample_weight=True,
    model_params={'n_estimators': 30, 'max_depth': 1, 'min_samples_leaf': 25},
)

v2_finder.fit(
    X_candidates,
    y=y_true,
    base_pred=oof_pred,
    sample_weight=sample_weight,
    original_model_features=base_feature_cols,
)

v2_summary = v2_finder.get_summary()
print('V2 top 10 features by residual lift:')
display(
    v2_summary[
        ['feature', 'mean_oof_residual_r2', 'null_beat_rate', 'action_category']
    ].head(10)
)

## Step 2 — Select interaction candidates

Keep features where `null_beat_rate > 0.60` (consistently beat their own permuted
baseline) and cap at 10 to keep the pairwise search fast. This gives C(10, 2) = 45
pairs × 50 bootstraps = 2 250 model fits — comfortable for a notebook.

In production you would increase `n_bootstraps` to 50+ and could expand to 15 features
if compute allows.

In [ ]:
MAX_INTERACTION_FEATURES = 10
NULL_BEAT_THRESHOLD = 0.60

credible = v2_summary.loc[
    v2_summary['null_beat_rate'].fillna(0) > NULL_BEAT_THRESHOLD
].copy()

interaction_candidates = credible.head(MAX_INTERACTION_FEATURES)['feature'].astype(str).tolist()
n_pairs = len(interaction_candidates) * (len(interaction_candidates) - 1) // 2

print(f'Credible features (null_beat_rate > {NULL_BEAT_THRESHOLD}): {len(credible)}')
print(f'Selected for interaction search: {len(interaction_candidates)} features → {n_pairs} pairs')
print('\nSelected features:')
for feat in interaction_candidates:
    row = v2_summary[v2_summary['feature'] == feat].iloc[0]
    print(f'  {feat:20s}  lift={row["mean_oof_residual_r2"]:.4f}  '
          f'null_beat_rate={row["null_beat_rate"]:.2f}')

## Step 3 — Fit InteractionFinder

`InteractionFinder` receives:
- `X`: the full feature matrix (any dtypes; only `candidate_features` are used).
- `residuals`: pre-computed `y_true − base_pred` — the same values V2 used internally.
- `candidate_features`: the short list selected above.
- `y_true` *(optional)*: the raw target labels. When provided, each interaction figure
  gains a **target-view panel** showing where actual target events concentrate in the
  same (feature_a, feature_b) space — making it easier to interpret what the interaction
  means for the outcome.

Categorical features are inferred from dtype (pd.Categorical or non-numeric) or can
be listed explicitly via `categorical_features=`. Here `age_band` is a string column
and will be detected automatically.

In [ ]:
finder = InteractionFinder(
    n_bootstraps=30,       # 50+ recommended in production; 30 used here for speed
    test_size=0.25,
    split_strategy='bootstrap',
    model_type='xgboost',
    random_state=42,
    model_params={
        'n_estimators': 80,
        'learning_rate': 0.05,
        'subsample': 0.8,
        'min_child_weight': 10,
        'reg_lambda': 2.0,
    },
)

finder.fit(
    X_candidates,
    residuals=residuals,
    candidate_features=interaction_candidates,
    y_true=y_true,          # enables the target-view panel in plot_interactions()
)

print('Detected categorical features:', finder._categorical_features)
print(f'Bootstrap results shape: {finder.interaction_bootstrap_results_.shape}')
print(f'Summary shape:           {finder.interaction_summary_.shape}')

## Interaction summary

The summary has one row per unique feature pair, sorted by `mean_interaction_lift`.

**Column guide:**

| Column | Meaning |
|---|---|
| `mean_interaction_lift` | Mean R²(depth-2) − R²(depth-1) across bootstraps. Positive = interaction adds predictive power beyond additive effects. |
| `median_interaction_lift` | Median version; less sensitive to outlier bootstrap splits. |
| `positive_lift_rate` | Fraction of bootstraps where lift > 0. Values near 0.5 indicate noise; > 0.70 suggests a consistent pattern. |
| `interaction_null_beat_rate` | Fraction of bootstraps where real lift > permuted-B null lift. > 0.75 is a credibility threshold. |
| `mean_depth2_r2` | Mean OOF R² of the depth-2 model. |
| `mean_depth1_r2` | Mean OOF R² of the depth-1 (additive) model. |
| `rank` | Ascending rank by `mean_interaction_lift`; rank 1 = highest lift. |

In [ ]:
display(finder.interaction_summary_)

## Credibility filter

Apply the same two-threshold filter used for V2 features: require `positive_lift_rate
> 0.60` and `interaction_null_beat_rate > 0.65` to surface pairs that consistently
show real interaction lift rather than noise.

In [ ]:
POSITIVE_LIFT_THRESHOLD = 0.60
NULL_BEAT_THRESHOLD_INTERACTION = 0.65

credible_pairs = finder.interaction_summary_.loc[
    (finder.interaction_summary_['positive_lift_rate'] > POSITIVE_LIFT_THRESHOLD)
    & (finder.interaction_summary_['interaction_null_beat_rate'] > NULL_BEAT_THRESHOLD_INTERACTION)
].copy()

print(f'Pairs passing credibility filter: {len(credible_pairs)} of {len(finder.interaction_summary_)}')
display(
    credible_pairs[
        [
            'feature_1', 'feature_2', 'rank',
            'mean_interaction_lift', 'positive_lift_rate', 'interaction_null_beat_rate',
        ]
    ]
)

## Per-bootstrap results

`interaction_bootstrap_results_` has one row per (feature_pair × bootstrap split).
Use it to inspect the distribution of lift and null lift for a specific pair.

In [ ]:
top_pair = finder.interaction_summary_.iloc[0]
f1, f2 = top_pair['feature_1'], top_pair['feature_2']

pair_runs = finder.interaction_bootstrap_results_.loc[
    (finder.interaction_bootstrap_results_['feature_1'] == f1)
    & (finder.interaction_bootstrap_results_['feature_2'] == f2)
].copy()

print(f'Top pair: {f1} × {f2}   (rank {int(top_pair["rank"])})')
print(f'  mean lift:          {pair_runs["interaction_lift"].mean():.5f}')
print(f'  mean null lift:     {pair_runs["null_lift"].mean():.5f}')
print(f'  positive_lift_rate: {pair_runs["beats_null"].mean():.2f}')
print()
display(pair_runs.head(10))

## Diagnostic plots

Each figure from `plot_interactions()` has **three panels**:

- **Left — Interaction effect**: how the depth-2 model prediction differs from the
  depth-1 (additive) prediction at each held-out observation. Red = positive interaction
  (the pair jointly predicts higher residuals than additively); blue = negative.
  - Continuous × continuous → scatter of feature_a vs feature_b, colored by net effect.
  - Continuous × categorical → conditional mean-residual curves, one line per category level.
  - Categorical × categorical → bubble chart; bubble color = mean net effect, bubble size = count.

- **Centre — Target distribution**: the same feature space, but colored by the actual
  `y_true` value. Comparing left and centre shows *why* the interaction matters: the
  pair's interaction effect (left) should align with where target events concentrate
  (centre) when the base model under-predicts. Requires `y_true=` in `fit()`.

- **Right — Lift distribution**: bootstrap interaction lift (blue) vs permuted-null lift
  (grey), with the null beat rate annotated.

In [ ]:
figures = finder.plot_interactions(top_n=5)

for pair_key, fig in figures.items():
    feat_a, feat_b = pair_key.split('__x__')
    row = finder.interaction_summary_.loc[
        ((finder.interaction_summary_['feature_1'] == feat_a) &
         (finder.interaction_summary_['feature_2'] == feat_b))
        | ((finder.interaction_summary_['feature_1'] == feat_b) &
           (finder.interaction_summary_['feature_2'] == feat_a))
    ].iloc[0]
    print(
        f'Pair {int(row["rank"])}: {feat_a} × {feat_b}   '
        f'lift={row["mean_interaction_lift"]:.5f}   '
        f'null_beat_rate={row["interaction_null_beat_rate"]:.2f}'
    )
    display(fig)
    plt.close(fig)

## Using user-supplied splits

`InteractionFinder` supports the same `splits=` API as V2: a list of dicts with
`train` / `validation` keys, or `(train_idx, val_idx)` tuples. This is useful when
you want the interaction analysis to use the same folds as the V2 run, or when you
have a fixed train/validation split from an existing pipeline.

In [ ]:
train_valid, holdout_rows = train_test_split(
    analysis_frame,
    test_size=0.15,
    stratify=analysis_frame[target_col],
    random_state=42,
)
train_rows, validation_rows = train_test_split(
    train_valid,
    test_size=0.1765,
    stratify=train_valid[target_col],
    random_state=42,
)

tvh_splits = [
    {
        'train': train_rows.index,
        'validation': validation_rows.index,
    }
]

# Use only the top 2 credible pairs for a quick demonstration
top_2_features = list(
    dict.fromkeys(
        finder.interaction_summary_.head(2)[['feature_1', 'feature_2']].to_numpy().flatten()
    )
)[:4]

custom_split_finder = InteractionFinder(n_bootstraps=1, random_state=0)
custom_split_finder.fit(
    X_candidates,
    residuals=residuals,
    candidate_features=top_2_features,
    splits=tvh_splits,
)

print(f'Bootstrap results rows (1 split × {len(top_2_features)*(len(top_2_features)-1)//2} pairs):')
display(custom_split_finder.interaction_bootstrap_results_)

## Interpreting results

**When is an interaction actionable?**

- `interaction_null_beat_rate ≥ 0.75` and `positive_lift_rate ≥ 0.65` with
  `mean_interaction_lift > 0.002` is a reasonable threshold for a feature pair worth
  engineering explicitly.
- A pair where `mean_depth1_r2` is already near zero may show a spuriously large
  *relative* lift. Always check the absolute depth-2 R² value.
- Interaction lift in residual space tells you whether a pair jointly predicts the base
  model's errors, not whether it will improve a retrained multivariate model. Use it
  as a signal to prioritise feature engineering, not as a direct performance guarantee.

**Next steps after identifying interactions:**

1. Engineer an explicit interaction feature (e.g., `pay_0 * bill_amt1`).
2. Re-run V2 on the augmented feature set to confirm the interaction feature captures
   additional residual signal.
3. Retrain the base model with the new feature and verify holdout performance improves.

**Computational cost guidance:**

| Candidate features | Pairs | Bootstraps | Total model fits |
|---|---|---|---|
| 5  | 10  | 50 | 2 000  |
| 10 | 45  | 50 | 9 000  |
| 15 | 105 | 50 | 21 000 |

Each fit trains 4 models (depth-1 real, depth-2 real, depth-1 null, depth-2 null).
Use `model_params` to reduce `n_estimators` when exploring, then increase for a final
run on the most promising pairs.